# Retraining the ubi-site score (Python)
Python port of `training_pipeline.Rmd`. Uses the functions in
`scripts/python/model_training.py`, `scripts/python/feature_processing.py`, and
`scripts/python/pipeline.py`.

In [ ]:
# Initialise
import sys

sys.path.append("scripts/python")

from feature_processing import prepare_features
from model_training import make_folds_protstrat
from pipeline import build_training_data, load_default_train_sites, run_training_pipeline

# Load data
`dat_feats` holds the processed features for every candidate site (index
`"<uniprot>_<site>"`), built from the raw per-lysine features in `allLys_HsRevi_feats.csv`
by `prepare_features()`. `train_sites` is a dict of the form
`{"pos": [...], "neg": [...]}`, giving the index values of `dat_feats` to use as the
positive/negative training sites, built by `load_default_train_sites()` from the sites
originally used to train the model.

In [ ]:
dat_feats = prepare_features("data")
train_sites = load_default_train_sites("data")

# Example 1: retrain with all the defaults

In [ ]:
res = run_training_pipeline(dat_feats, train_sites)

res["auc_df"]           # AUC of each train-test split, on its own held-out test set
res["test_preds_df"]    # the underlying test-set predictions behind auc_df
res["all_preds_df"]     # median predicted score per site in dat_feats, across all splits

# Example 2: custom training sites and repeated splits
Supply your own positive/negative training sites (as index values of `dat_feats`), and
generate 5 repeats of the 5-fold split instead of 1.

In [ ]:
my_train_sites = {
    "pos": train_sites["pos"],
    "neg": train_sites["neg"][:5000],
}

res_custom = run_training_pipeline(dat_feats, my_train_sites, num_reps=5)

# Example 3: prepare_features() with custom sites
`sites` can be any list of `uniprot_site` values representing lysines in the reviewed human proteome, not just the ones already used for
training - e.g. adding a couple of extra sites of interest alongside `train_sites`:

In [ ]:
dat_feats_extra_sites = prepare_features(
    "data",
    sites= list(set(["Q9NVI1_K523", "Q9BXW9_K561"] + train_sites["pos"] + train_sites["neg"]))
)

# Example 4: supplying your own train-test splits
If you'd rather build the train-test splits yourself (e.g. a different splitting
strategy), build a training set with `build_training_data()`, construct a dict of
`{"train_dat": ..., "test_dat": ...}` splits from it, and pass that in directly.

In [ ]:
train_dat = build_training_data(dat_feats, my_train_sites["pos"], my_train_sites["neg"])
my_custom_splits = make_folds_protstrat(train_dat, k=5, seed=1)  # or your own splitting logic

res_custom_splits = run_training_pipeline(
    dat_feats, my_train_sites, custom_splits=my_custom_splits
)

# Example 5: customising feature processing
`prepare_features()` runs: subset to `sites` (default: `load_default_pred_sites()`) ->
subset to `features` -> mean-impute `mean_impute_feats` (default `FinalBoxProcNum`) ->
drop sites with any remaining missing feature values -> log-transform `log_feats` ->
center and scale.

In [ ]:
# Mean-impute an extra feature
dat_feats_custom_impute = prepare_features("data", mean_impute_feats=["FinalBoxProcNum", "NSP_rsa"])

In [ ]:
# Restrict processing to a chosen set of sites, e.g. only the sites used for training
dat_feats_subset = prepare_features("data", sites=train_sites["pos"] + train_sites["neg"])

In [ ]:
# Select only a subset of features
dat_feats_subset_feats = prepare_features("data", features=["FinalBoxProcNum", "NSP_rsa", "PRIDE_gold"])

In [ ]:
# Skip processing entirely and get the raw features (still subsetted to features/sites)
dat_feats_raw = prepare_features("data", raw=True)